# Development of Standalone Models with Rolling Validation: SARIMAX, LSTM and XGBoost

Builds the three standalone benchmark models and evaluates them with **expanding-window rolling-origin validation** over the model-development period (2005–2022).

## Forecasting task: year-ahead forecast
Every model is trained on **monthly** data. At each forecast origin (**December of year Y−1**), it forecasts the **12 monthly GHI values of year Y** (horizons 1–12 months). The annual mean GHI is the mean of these 12 values.

Only information available at the forecast origin is used:

| Model | Scope | Inputs |
|---|---|---|
| SARIMAX | one model per grid | past GHI (AR/MA terms) + cloud amount and RH of the **same month one year earlier** (exogenous, lag 12), which are known at the origin for all 12 horizons |
| LSTM | one global model | 12-month window ending at the origin: GHI, cloud amount, RH, month encoding, grid lat/lon/elevation → 12 outputs (multi-output) |
| XGBoost | one global model | values at lags 0–11 months before the origin of GHI, cloud amount and RH, horizon, target month, grid lat/lon/elevation (direct multi-horizon) |
| Climatology | reference | mean GHI of the same calendar month over all training years |
| Seasonal naive | reference | GHI of the same month in year Y−1 |

## Data periods

| Period | Years | Use |
|---|---|---|
| Model development + rolling validation | 2005–2022 | this notebook |
| Final testing | 2023–2024 | **held out**, removed on load |
| Final product | retrain to 2025, forecast 2026 | later step |

## Rolling validation (expanding window, 8 rounds)

| Round | Training | Forecast origin | Validation year |
|---|---|---|---|
| 1 | 2005–2014 | Dec 2014 | 2015 |
| 2 | 2005–2015 | Dec 2015 | 2016 |
| … | … | … | … |
| 8 | 2005–2021 | Dec 2021 | 2022 |

In each round, models are trained only on data up to the forecast origin and then forecast the next year. The first training window (10 years) gives SARIMAX enough seasonal cycles, and the validation years cover the 2015–16 El Niño and the 2020–22 La Niña.

- **SARIMAX:** orders are selected once on the round-1 training window; the parameters are re-estimated every round.
- **LSTM and XGBoost:** retrained from scratch every round. The last training year is used only for early stopping.

## Outputs (`outputs/standalone_models/`)
- `rolling_predictions/<model>_rolling_predictions.csv`: every forecast (round, grid, target month, actual, predicted)
- `rolling_metrics/<model>_rolling_metrics_by_grid.csv`: **RMSE, MAE, nRMSE per round and grid point**
- `rolling_metrics/<model>_rolling_metrics_by_round.csv`: RMSE, MAE, nRMSE per round (all grids pooled)
- `rolling_metrics/rolling_metrics_*_all_models.csv` and `rolling_metrics_summary.csv`: all models combined

## Setup
```
%pip install statsmodels scikit-learn xgboost torch joblib
```
Run from the `Philippine Spatial Map` folder. Run the setup cell first. Each model section is independent, and the final section compares whichever models have been run.

In [1]:
"""
standalone_models_rolling.py

Purpose:
    Common setup for the standalone year-ahead GHI models:
    data loading, rolling-validation rounds, error metrics
    and output helpers.

Input:
    data/processed/nasa_power_validated.csv
    data/raw/elevation/grid_elevation.csv

Output:
    outputs/standalone_models/
"""

from pathlib import Path
from importlib import metadata
import platform
import random
import warnings

import numpy as np
import pandas as pd


# ============================================================
# 1. USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# INPUT FILES
# ------------------------------------------------------------

DATA_FILE = Path(
    "data/processed/nasa_power_validated.csv"
)

ELEVATION_FILE = Path(
    "data/raw/elevation/grid_elevation.csv"
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "outputs/standalone_models"
)

PREDICTION_DIR = OUTPUT_DIR / "rolling_predictions"

METRICS_DIR = OUTPUT_DIR / "rolling_metrics"

MODEL_DIR = OUTPUT_DIR / "models"


# ------------------------------------------------------------
# VARIABLES
# ------------------------------------------------------------

GHI_COL = "ghi_kwh_m2_day"

CLOUD_COL = "cloud_amt_pct"

RH_COL = "rh2m_pct"

VARIABLES = [
    GHI_COL,
    CLOUD_COL,
    RH_COL
]

EXOG_COLS = [
    CLOUD_COL,
    RH_COL
]

STATIC_COLS = [
    "latitude",
    "longitude",
    "elevation_m"
]

CALENDAR_COLS = [
    "month_sin",
    "month_cos"
]


# ------------------------------------------------------------
# MODEL-DEVELOPMENT PERIOD
# ------------------------------------------------------------
# 2023-2024 is reserved for final testing and removed on load.

DEVELOPMENT_START = "2005-01-01"

DEVELOPMENT_END = "2022-12-01"


# ------------------------------------------------------------
# ROLLING VALIDATION (expanding window)
# ------------------------------------------------------------
# Round 1 trains on 2005-2014 and validates 2015;
# the last round trains on 2005-2021 and validates 2022.

FIRST_VALIDATION_YEAR = 2015

LAST_VALIDATION_YEAR = 2022


# ------------------------------------------------------------
# FORECAST SETTINGS
# ------------------------------------------------------------

# Year-ahead: 12 monthly values from a December origin
HORIZON = 12

# Months of history given to the LSTM and XGBoost
WINDOW = 12

SEASONAL_PERIOD = 12


# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

SEED = 42


# ============================================================
# 2. CREATE OUTPUT DIRECTORIES
# ============================================================

for directory in [PREDICTION_DIR, METRICS_DIR, MODEL_DIR]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

def set_seed(seed=SEED):
    """Fix the Python and NumPy random seeds."""

    random.seed(seed)

    np.random.seed(seed)


set_seed()


def library_versions():
    """Versions of every library used, for the paper's reproducibility section."""

    versions = {"python": platform.python_version()}

    for package in ["numpy", "pandas", "statsmodels", "scikit-learn", "torch", "xgboost", "joblib"]:

        try:
            versions[package] = metadata.version(package)
        except metadata.PackageNotFoundError:
            versions[package] = "not installed"

    return versions


VERSIONS = library_versions()

(OUTPUT_DIR / "environment.txt").write_text(
    "\n".join(f"{name}=={version}" for name, version in VERSIONS.items()),
    encoding="utf-8"
)

print("Library versions:")

for name, version in VERSIONS.items():
    print(f"  {name:<13} {version}")


# ============================================================
# 4. LOAD DATA
# ============================================================

print("\nLoading data...")

data = pd.read_csv(
    DATA_FILE,
    parse_dates=["date"]
)

elevation = pd.read_csv(
    ELEVATION_FILE
)[["grid_number", "elevation_m"]]

data = (
    data.merge(elevation, on="grid_number", how="left")
    .sort_values(["grid_number", "date"])
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Keep the model-development period only
# ------------------------------------------------------------

n_before = len(data)

data = data[
    (data["date"] >= DEVELOPMENT_START)
    & (data["date"] <= DEVELOPMENT_END)
].reset_index(drop=True)

print(
    f"Held out {n_before - len(data)} records after {DEVELOPMENT_END[:7]} "
    f"(final testing)"
)


# ------------------------------------------------------------
# Fill any values set to NaN during validation
# (linear interpolation within each grid)
# ------------------------------------------------------------

n_missing = int(data[VARIABLES].isna().sum().sum())

if n_missing:

    print(f"Interpolating {n_missing} missing values within each grid")

    data[VARIABLES] = (
        data.groupby("grid_number")[VARIABLES]
        .transform(lambda s: s.interpolate(limit_direction="both"))
    )


# ------------------------------------------------------------
# Cyclical month encoding
# ------------------------------------------------------------

data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)

data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)


GRID_INFO = (
    data[["grid_number", "grid_id", "latitude", "longitude"]]
    .drop_duplicates("grid_number")
    .reset_index(drop=True)
)

N_GRIDS = len(GRID_INFO)

print(
    f"Grids: {N_GRIDS}   "
    f"Months: {data['date'].min():%Y-%m} to {data['date'].max():%Y-%m}   "
    f"Records: {len(data)}"
)


# ============================================================
# 5. ROLLING-VALIDATION ROUNDS
# ============================================================

ROUNDS = pd.DataFrame([
    {
        "round":            number,
        "validation_year":  year,
        "train_start":      DEVELOPMENT_START[:7],
        "train_end":        f"{year - 1}-12",
        "train_end_date":   pd.Timestamp(f"{year - 1}-12-01"),
        "validation_start": pd.Timestamp(f"{year}-01-01"),
        "validation_end":   pd.Timestamp(f"{year}-12-01")
    }
    for number, year in enumerate(
        range(FIRST_VALIDATION_YEAR, LAST_VALIDATION_YEAR + 1),
        start=1
    )
])

ROUND_COLS = [
    "round",
    "validation_year",
    "train_start",
    "train_end"
]

GRID_COLS = [
    "grid_number",
    "grid_id",
    "latitude",
    "longitude"
]

ROUNDS[ROUND_COLS].to_csv(
    OUTPUT_DIR / "rolling_rounds.csv",
    index=False
)

print("\nRolling-validation rounds:")

print(
    ROUNDS[ROUND_COLS]
    .assign(forecast_origin=ROUNDS["train_end"])
    .to_string(index=False)
)


# ============================================================
# 6. ERROR METRICS
# ============================================================

def summarise_errors(frame, keys):
    """
    RMSE, MAE (kWh/m²/day) and nRMSE (% of mean observed GHI)
    for every group defined by `keys`.

    mean_bias_pct compares the mean forecast with the mean
    observation; per grid and round this is the error of the
    forecast annual-mean GHI.
    """

    error = frame["predicted"] - frame["actual"]

    summary = (
        frame.assign(
            squared_error=error ** 2,
            absolute_error=error.abs()
        )
        .groupby(keys, sort=True)
        .agg(
            RMSE=("squared_error", "mean"),
            MAE=("absolute_error", "mean"),
            mean_actual=("actual", "mean"),
            mean_predicted=("predicted", "mean"),
            n_months=("actual", "size")
        )
        .reset_index()
    )

    summary["RMSE"] = np.sqrt(summary["RMSE"])

    summary["nRMSE_pct"] = summary["RMSE"] / summary["mean_actual"] * 100

    summary["mean_bias_pct"] = (
        (summary["mean_predicted"] - summary["mean_actual"])
        / summary["mean_actual"] * 100
    )

    return summary[
        keys
        + ["RMSE", "MAE", "nRMSE_pct", "mean_actual", "mean_predicted", "mean_bias_pct", "n_months"]
    ]


# ============================================================
# 7. OUTPUT HELPER
# ============================================================

def save_rolling_results(predictions, name, label):
    """
    Attach actuals, round and grid information to a model's
    rolling forecasts, then save:
        rolling_predictions/<name>_rolling_predictions.csv
        rolling_metrics/<name>_rolling_metrics_by_grid.csv
        rolling_metrics/<name>_rolling_metrics_by_round.csv

    `predictions` needs the columns: round, grid_number,
    target_date, predicted.
    """

    frame = (
        predictions[["round", "grid_number", "target_date", "predicted"]]
        .merge(ROUNDS[ROUND_COLS], on="round", how="left")
        .merge(
            data[["grid_number", "date", GHI_COL]]
            .rename(columns={"date": "target_date", GHI_COL: "actual"}),
            on=["grid_number", "target_date"],
            how="left"
        )
        .merge(GRID_INFO, on="grid_number", how="left")
    )

    frame["model"] = label

    # Origin is December of the previous year, so horizon = target month
    frame["horizon_months"] = frame["target_date"].dt.month

    frame["error"] = frame["predicted"] - frame["actual"]


    # ----------------------------------------------------
    # Completeness checks
    # ----------------------------------------------------

    expected_rows = len(ROUNDS) * N_GRIDS * HORIZON

    wrong_year = frame["target_date"].dt.year != frame["validation_year"]

    if (
        len(frame) != expected_rows
        or frame[["actual", "predicted"]].isna().any().any()
        or wrong_year.any()
    ):

        raise ValueError(
            f"{label}: {len(frame)} forecasts (expected {expected_rows}), "
            f"{frame['predicted'].isna().sum()} missing predictions, "
            f"{wrong_year.sum()} outside their validation year"
        )


    frame = frame[
        ["model"] + ROUND_COLS + GRID_COLS
        + ["target_date", "horizon_months", "actual", "predicted", "error"]
    ].sort_values(["round", "grid_number", "target_date"]).reset_index(drop=True)

    by_grid = summarise_errors(frame, ["model"] + ROUND_COLS + GRID_COLS)

    by_round = summarise_errors(frame, ["model"] + ROUND_COLS)


    frame.to_csv(PREDICTION_DIR / f"{name}_rolling_predictions.csv", index=False)

    by_grid.to_csv(METRICS_DIR / f"{name}_rolling_metrics_by_grid.csv", index=False)

    by_round.to_csv(METRICS_DIR / f"{name}_rolling_metrics_by_round.csv", index=False)


    overall = summarise_errors(frame, ["model"]).iloc[0]

    print(f"\n{label}: rolling validation by round (all grids pooled)")

    print(
        by_round[["round", "validation_year", "RMSE", "MAE", "nRMSE_pct"]]
        .round(4)
        .to_string(index=False)
    )

    print(
        f"All rounds: RMSE {overall['RMSE']:.4f}   MAE {overall['MAE']:.4f}   "
        f"nRMSE {overall['nRMSE_pct']:.2f}%"
    )

    print(f"Saved: {PREDICTION_DIR / f'{name}_rolling_predictions.csv'}")
    print(f"Saved: {METRICS_DIR / f'{name}_rolling_metrics_by_grid.csv'}")
    print(f"Saved: {METRICS_DIR / f'{name}_rolling_metrics_by_round.csv'}")

    return frame, by_grid, by_round

Library versions:
  python        3.14.7
  numpy         2.5.3
  pandas        3.0.6
  statsmodels   0.15.0
  scikit-learn  1.9.1
  torch         2.14.0
  xgboost       3.4.1
  joblib        1.6.0

Loading data...
Held out 2232 records after 2022-12 (final testing)
Grids: 93   Months: 2005-01 to 2022-12   Records: 20088

Rolling-validation rounds:
 round  validation_year train_start train_end forecast_origin
     1             2015     2005-01   2014-12         2014-12
     2             2016     2005-01   2015-12         2015-12
     3             2017     2005-01   2016-12         2016-12
     4             2018     2005-01   2017-12         2017-12
     5             2019     2005-01   2018-12         2018-12
     6             2020     2005-01   2019-12         2019-12
     7             2021     2005-01   2020-12         2020-12
     8             2022     2005-01   2021-12         2021-12


## 1. SARIMAX

Seasonal ARIMA with exogenous regressors, fitted **separately for each grid cell** with the statsmodels state-space implementation.

- **Exogenous regressors:** cloud amount and RH **lagged 12 months**. The forecast for month *m* of year Y uses cloud amount and RH of month *m* in year Y−1, which are known at the December origin, so all 12 horizons can be forecast without assuming future weather.
- **Seasonal differencing:** D = 1, with period s = 12.
- **Non-seasonal differencing d:** the ADF test is run on the seasonally differenced training series. If p < 0.05, d = 0; otherwise d = 1.
- **Order selection:** AIC grid search over p, q ∈ {0, 1, 2} and P, Q ∈ {0, 1} (36 candidates), done **once** on the round-1 training window (2005–2014).
- **Each round:** the parameters are re-estimated on that round's training data by maximum likelihood (L-BFGS, up to 200 iterations, warm-started from the previous round's parameters), then 12 months are forecast from the December origin (a dynamic multi-step forecast).
- **Diagnostics:** a Ljung-Box test at lag 12 and the optimiser's convergence flag are recorded for every grid and round.

Grids run in parallel with joblib, which took about 5 minutes on a 12-core CPU.

In [2]:
# ============================================================
# 1. SARIMAX (per grid)
# ============================================================

from itertools import product

from joblib import Parallel, delayed
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller


# ------------------------------------------------------------
# SARIMAX SETTINGS
# ------------------------------------------------------------

# Exogenous variables lagged one year (known at the origin)
EXOG_LAG = 12

P_VALUES = [0, 1, 2]

Q_VALUES = [0, 1, 2]

SEASONAL_P_VALUES = [0, 1]

SEASONAL_Q_VALUES = [0, 1]

SEASONAL_D = 1

ADF_ALPHA = 0.05

# Maximum optimiser (L-BFGS) iterations; the statsmodels default of 50
# leaves many seasonal models unconverged
MAX_ITER = 200

# Number of parallel workers (-1 = all CPU cores)
N_JOBS = -1


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def fit_sarimax(endog, exog, order, seasonal_order, start_params=None):
    """Fit one SARIMAX model, silencing optimiser warnings."""

    with warnings.catch_warnings():

        warnings.simplefilter("ignore")

        return SARIMAX(
            endog,
            exog=exog,
            order=order,
            seasonal_order=seasonal_order
        ).fit(
            start_params=start_params,
            maxiter=MAX_ITER,
            disp=False
        )


def select_sarimax_order(endog_train, exog_train):
    """
    Choose d with the ADF test, then (p, q)(P, Q) by AIC.
    """

    seasonal_diff = endog_train.diff(SEASONAL_PERIOD).dropna()

    with warnings.catch_warnings():

        # statsmodels >= 0.15 warns about a future change of return type
        warnings.simplefilter("ignore", FutureWarning)

        adf_result = adfuller(seasonal_diff, autolag="AIC")

    adf_pvalue = adf_result[1] if isinstance(adf_result, tuple) else adf_result.pvalue

    d = 0 if adf_pvalue < ADF_ALPHA else 1


    best_aic = np.inf

    best_order = None

    best_seasonal = None

    best_result = None

    n_fitted = 0

    for p, q, P, Q in product(P_VALUES, Q_VALUES, SEASONAL_P_VALUES, SEASONAL_Q_VALUES):

        order = (p, d, q)

        seasonal_order = (P, SEASONAL_D, Q, SEASONAL_PERIOD)

        try:
            result = fit_sarimax(endog_train, exog_train, order, seasonal_order)
        except Exception:
            continue

        if not np.isfinite(result.aic):
            continue

        n_fitted += 1

        if result.aic < best_aic:

            best_aic = result.aic

            best_order = order

            best_seasonal = seasonal_order

            best_result = result


    if best_order is None:

        raise RuntimeError("No SARIMAX candidate could be fitted")


    selection = {
        "adf_pvalue":        adf_pvalue,
        "selection_aic":     best_aic,
        "candidates_fitted": n_fitted
    }

    return best_order, best_seasonal, selection, best_result


# ------------------------------------------------------------
# One grid: select order once, then refit and forecast each round
# ------------------------------------------------------------

def run_sarimax_grid(grid_number, frame):

    series = (
        frame.set_index("date")
        .asfreq("MS")
    )

    # Drop the first year, which has no lagged exogenous values
    exog = series[EXOG_COLS].shift(EXOG_LAG).iloc[EXOG_LAG:]

    endog = series[GHI_COL].iloc[EXOG_LAG:]


    first_train_end = ROUNDS["train_end_date"].iloc[0]

    order, seasonal_order, selection, selected_result = select_sarimax_order(
        endog.loc[:first_train_end],
        exog.loc[:first_train_end]
    )


    predictions = []

    diagnostics = []

    for rnd in ROUNDS.itertuples(index=False):

        if rnd.round == 1:

            # Round 1 uses the same training window as order selection
            result = selected_result

        else:

            # Warm start from the previous round's parameters
            result = fit_sarimax(
                endog.loc[:rnd.train_end_date],
                exog.loc[:rnd.train_end_date],
                order,
                seasonal_order,
                start_params=result.params
            )

        forecast = result.get_forecast(
            steps=HORIZON,
            exog=exog.loc[rnd.validation_start:rnd.validation_end]
        ).predicted_mean

        burn_in = order[1] + SEASONAL_D * SEASONAL_PERIOD

        ljung_box_p = acorr_ljungbox(
            result.resid.iloc[burn_in:],
            lags=[SEASONAL_PERIOD]
        )["lb_pvalue"].iloc[0]

        predictions.append(pd.DataFrame({
            "round":       rnd.round,
            "grid_number": grid_number,
            "target_date": forecast.index,
            "predicted":   forecast.to_numpy()
        }))

        diagnostics.append({
            "round":             rnd.round,
            "validation_year":   rnd.validation_year,
            "grid_number":       grid_number,
            "order":             str(order),
            "seasonal_order":    str(seasonal_order),
            "aic":               result.aic,
            "converged":         bool(result.mle_retvals.get("converged", True)),
            "ljung_box_p_lag12": ljung_box_p,
            **selection
        })

    return pd.concat(predictions, ignore_index=True), diagnostics


# ------------------------------------------------------------
# Run all grids in parallel
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SARIMAX: ORDER SELECTION AND ROLLING VALIDATION")
print("=" * 65)

print(
    f"Grids: {N_GRIDS}   Rounds: {len(ROUNDS)}   "
    f"Exogenous: {EXOG_COLS} (lag {EXOG_LAG})"
)

sarimax_outputs = Parallel(n_jobs=N_JOBS, verbose=5)(
    delayed(run_sarimax_grid)(grid_number, frame)
    for grid_number, frame in data.groupby("grid_number")
)

sarimax_predictions = pd.concat(
    [output[0] for output in sarimax_outputs],
    ignore_index=True
)

sarimax_diagnostics = pd.DataFrame(
    [row for output in sarimax_outputs for row in output[1]]
)

sarimax_diagnostics.to_csv(
    MODEL_DIR / "sarimax_rolling_diagnostics.csv",
    index=False
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

first_round = sarimax_diagnostics[sarimax_diagnostics["round"] == 1]

print("\nSelected orders (count of grids):")

print(
    first_round.groupby(["order", "seasonal_order"])
    .size()
    .sort_values(ascending=False)
    .to_string()
)

print(f"\nGrids with d = 1: {(first_round['adf_pvalue'] >= ADF_ALPHA).sum()}")

print(
    f"Fits not converged: {(~sarimax_diagnostics['converged']).sum()} "
    f"of {len(sarimax_diagnostics)}"
)

print(
    f"Fits with autocorrelated residuals (Ljung-Box p < 0.05): "
    f"{(sarimax_diagnostics['ljung_box_p_lag12'] < 0.05).sum()} "
    f"of {len(sarimax_diagnostics)}"
)

print(f"Diagnostics saved: {MODEL_DIR / 'sarimax_rolling_diagnostics.csv'}")

sarimax_results = save_rolling_results(
    sarimax_predictions,
    "sarimax",
    "SARIMAX"
)


SARIMAX: ORDER SELECTION AND ROLLING VALIDATION
Grids: 93   Rounds: 8   Exogenous: ['cloud_amt_pct', 'rh2m_pct'] (lag 12)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:  2.9min
[Parallel(n_jobs=-1)]: Done  89 out of  93 | elapsed:  5.1min remaining:   13.8s
[Parallel(n_jobs=-1)]: Done  93 out of  93 | elapsed:  5.2min finished



Selected orders (count of grids):
order      seasonal_order
(2, 0, 2)  (0, 1, 1, 12)     26
(1, 0, 1)  (0, 1, 1, 12)     16
(0, 0, 0)  (0, 1, 1, 12)     16
(1, 0, 2)  (0, 1, 1, 12)     10
(0, 1, 1)  (0, 1, 1, 12)      5
(2, 0, 0)  (0, 1, 1, 12)      5
(2, 1, 1)  (0, 1, 1, 12)      4
(1, 0, 0)  (0, 1, 1, 12)      3
(0, 0, 0)  (1, 1, 1, 12)      2
(2, 1, 2)  (0, 1, 1, 12)      2
(2, 0, 2)  (1, 1, 1, 12)      2
(1, 1, 2)  (0, 1, 1, 12)      1
(2, 1, 0)  (0, 1, 1, 12)      1

Grids with d = 1: 13
Fits not converged: 4 of 744
Fits with autocorrelated residuals (Ljung-Box p < 0.05): 94 of 744
Diagnostics saved: outputs\standalone_models\models\sarimax_rolling_diagnostics.csv

SARIMAX: rolling validation by round (all grids pooled)
 round  validation_year   RMSE    MAE  nRMSE_pct
     1             2015 0.5414 0.4607    10.1192
     2             2016 0.5317 0.4097    10.0455
     3             2017 0.3376 0.2741     6.7645
     4             2018 0.4292 0.3301     8.3167
     5             

## 2. LSTM

A **global** Long Short-Term Memory network, trained on all grids together and implemented in PyTorch, using a **multi-output** (MIMO) design: one forward pass produces all 12 monthly forecasts.

- **Input:** a 12-month window ending at the origin. At each time step the features are GHI, cloud amount and RH (min-max scaled), sin/cos of the month, and the grid's latitude, longitude and elevation (scaled).
- **Output:** 12 values (scaled GHI for the next 12 months), converted back to kWh/m²/day.
- **Training samples:** every month is used as an origin, not only December, to give the network enough training examples. Validation uses only the December origin.
- **Per round:**
  - The scaler is fitted on that round's training rows only.
  - The model is fitted on samples whose last target is before the final training year.
  - Samples whose targets end in the final training year are used for early stopping (patience 20, best weights restored).
- **Architecture:** 2 stacked LSTM layers (64 units) → dropout (0.2) → a linear layer with 12 outputs.
- **Optimiser:** Adam (learning rate 1e-3), MSE loss, batch size 64, up to 200 epochs.

Retrained every round, this section took about 9 minutes on a CPU. XGBoost takes about 1–2 minutes.

In [3]:
# ============================================================
# 2. LSTM (global multi-output model, PyTorch)
# ============================================================

import copy

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler


# ------------------------------------------------------------
# LSTM SETTINGS
# ------------------------------------------------------------

HIDDEN_SIZE = 64

NUM_LAYERS = 2

DROPOUT = 0.2

LEARNING_RATE = 1e-3

BATCH_SIZE = 64

MAX_EPOCHS = 200

PATIENCE = 20


# Scaled columns (GHI is used for inverse scaling)
SCALED_COLS = VARIABLES + STATIC_COLS

LSTM_FEATURES = SCALED_COLS + CALENDAR_COLS

GHI_INDEX = SCALED_COLS.index(GHI_COL)

torch.use_deterministic_algorithms(True)


# ------------------------------------------------------------
# Sequences: 12-month input window -> next 12 months
# ------------------------------------------------------------

def make_sequences(frame, feature_cols, window, horizon):
    """
    For every grid and origin month i, build
        X = features of months i-window+1 ... i
        Y = GHI of months i+1 ... i+horizon
    """

    X, Y, grid_numbers, origins = [], [], [], []

    for grid_number, group in frame.groupby("grid_number"):

        features = group[feature_cols].to_numpy(dtype=np.float32)

        target = group[GHI_COL].to_numpy(dtype=np.float32)

        dates = group["date"].to_numpy()

        for i in range(window - 1, len(group) - horizon):

            X.append(features[i - window + 1:i + 1])

            Y.append(target[i + 1:i + 1 + horizon])

            grid_numbers.append(grid_number)

            origins.append(dates[i])

    keys = pd.DataFrame({
        "grid_number": grid_numbers,
        "origin_date": pd.to_datetime(origins)
    })

    return np.stack(X), np.stack(Y), keys


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

class LSTMForecaster(nn.Module):
    """Stacked LSTM with a multi-output linear head."""

    def __init__(self, n_features, hidden_size, num_layers, dropout, horizon):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)

        self.output = nn.Linear(hidden_size, horizon)

    def forward(self, x):

        sequence_output, _ = self.lstm(x)

        return self.output(self.dropout(sequence_output[:, -1, :]))


def train_lstm(X_fit, Y_fit, X_es, Y_es, round_number):
    """Train with early stopping; return the best model and history."""

    set_seed(SEED)

    torch.manual_seed(SEED)

    model = LSTMForecaster(
        n_features=X_fit.shape[2],
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
        horizon=HORIZON
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    loss_function = nn.MSELoss()

    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_fit), torch.from_numpy(Y_fit)),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED)
    )

    X_es = torch.from_numpy(X_es)

    Y_es = torch.from_numpy(Y_es)


    best_loss = np.inf

    best_state = None

    best_epoch = 0

    wait = 0

    history = []

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()

        running_loss = 0.0

        for X_batch, Y_batch in loader:

            optimizer.zero_grad()

            loss = loss_function(model(X_batch), Y_batch)

            loss.backward()

            optimizer.step()

            running_loss += loss.item() * len(X_batch)

        model.eval()

        with torch.no_grad():
            es_loss = loss_function(model(X_es), Y_es).item()

        history.append({
            "round": round_number,
            "epoch": epoch,
            "train_mse": running_loss / len(X_fit),
            "early_stop_mse": es_loss
        })

        if es_loss < best_loss:

            best_loss = es_loss

            best_state = copy.deepcopy(model.state_dict())

            best_epoch = epoch

            wait = 0

        else:

            wait += 1

            if wait >= PATIENCE:
                break

    model.load_state_dict(best_state)

    model.eval()

    return model, best_epoch, best_loss, history


# ------------------------------------------------------------
# Rolling validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("LSTM: ROLLING VALIDATION")
print("=" * 65)

lstm_predictions = []

lstm_history = []

for rnd in ROUNDS.itertuples(index=False):

    # Data up to the end of the validation year
    frame = data[data["date"] <= rnd.validation_end]

    # Scaler fitted on this round's training rows only
    scaler = MinMaxScaler().fit(
        frame.loc[frame["date"] <= rnd.train_end_date, SCALED_COLS]
    )

    scaled = frame.copy()

    scaled[SCALED_COLS] = scaler.transform(frame[SCALED_COLS])

    X, Y, keys = make_sequences(scaled, LSTM_FEATURES, WINDOW, HORIZON)


    last_target = keys["origin_date"] + pd.DateOffset(months=HORIZON)

    early_stop_start = rnd.train_end_date - pd.DateOffset(months=11)

    fit_mask = (last_target < early_stop_start).to_numpy()

    es_mask = (
        (last_target >= early_stop_start)
        & (last_target <= rnd.train_end_date)
    ).to_numpy()

    val_mask = (keys["origin_date"] == rnd.train_end_date).to_numpy()


    model, best_epoch, best_loss, history = train_lstm(
        X[fit_mask], Y[fit_mask],
        X[es_mask], Y[es_mask],
        rnd.round
    )

    lstm_history.extend(history)


    with torch.no_grad():
        predicted_scaled = model(torch.from_numpy(X[val_mask])).numpy()

    predicted = (
        predicted_scaled * scaler.data_range_[GHI_INDEX]
        + scaler.data_min_[GHI_INDEX]
    )

    val_grids = keys.loc[val_mask, "grid_number"].to_numpy()

    target_dates = pd.date_range(rnd.validation_start, periods=HORIZON, freq="MS")

    lstm_predictions.append(pd.DataFrame({
        "round":       rnd.round,
        "grid_number": np.repeat(val_grids, HORIZON),
        "target_date": np.tile(target_dates, len(val_grids)),
        "predicted":   predicted.ravel()
    }))

    torch.save(
        {
            "state_dict": model.state_dict(),
            "features": LSTM_FEATURES,
            "scaled_cols": SCALED_COLS,
            "scaler_min": scaler.data_min_.tolist(),
            "scaler_max": scaler.data_max_.tolist(),
            "config": {
                "round": rnd.round,
                "train_end": rnd.train_end,
                "window": WINDOW,
                "horizon": HORIZON,
                "hidden_size": HIDDEN_SIZE,
                "num_layers": NUM_LAYERS,
                "dropout": DROPOUT,
                "learning_rate": LEARNING_RATE,
                "batch_size": BATCH_SIZE,
                "best_epoch": best_epoch,
                "seed": SEED
            }
        },
        MODEL_DIR / f"lstm_round{rnd.round}.pt"
    )

    print(
        f"Round {rnd.round} (validate {rnd.validation_year}): "
        f"fit {fit_mask.sum()}  early-stop {es_mask.sum()}  "
        f"best epoch {best_epoch}  early-stop MSE {best_loss:.5f}"
    )


pd.DataFrame(lstm_history).to_csv(
    MODEL_DIR / "lstm_training_history.csv",
    index=False
)

lstm_results = save_rolling_results(
    pd.concat(lstm_predictions, ignore_index=True),
    "lstm",
    "LSTM"
)


LSTM: ROLLING VALIDATION
Round 1 (validate 2015): fit 7905  early-stop 1116  best epoch 35  early-stop MSE 0.00565
Round 2 (validate 2016): fit 9021  early-stop 1116  best epoch 44  early-stop MSE 0.00551
Round 3 (validate 2017): fit 10137  early-stop 1116  best epoch 29  early-stop MSE 0.00803
Round 4 (validate 2018): fit 11253  early-stop 1116  best epoch 13  early-stop MSE 0.00431
Round 5 (validate 2019): fit 12369  early-stop 1116  best epoch 36  early-stop MSE 0.00512
Round 6 (validate 2020): fit 13485  early-stop 1116  best epoch 11  early-stop MSE 0.00624
Round 7 (validate 2021): fit 14601  early-stop 1116  best epoch 21  early-stop MSE 0.00500
Round 8 (validate 2022): fit 15717  early-stop 1116  best epoch 7  early-stop MSE 0.00615

LSTM: rolling validation by round (all grids pooled)
 round  validation_year   RMSE    MAE  nRMSE_pct
     1             2015 0.5231 0.4369     9.7776
     2             2016 0.4709 0.3499     8.8956
     3             2017 0.4081 0.3342     8.1761

## 3. XGBoost

A **global** gradient-boosted tree model, trained on all grids together, using the **direct multi-horizon** strategy: one model covers all 12 horizons, with the horizon given as an input feature.

- **Features (42):**
  - GHI, cloud amount and RH at lags 0–11 months before the origin (36 features; lag 0 is the origin month)
  - horizon (1–12)
  - sin/cos of the target month
  - the grid's latitude, longitude and elevation
- **Training samples:** one row per grid × origin month × horizon. Rows whose target falls before the final training year are used for fitting. Rows whose target falls in the final training year are used for early stopping (100 rounds).
- **Validation:** only the December origin (horizons 1–12 = January–December of the validation year).
- Tree models don't need scaling, so the features are used in their original units.

In [4]:
# ============================================================
# 3. XGBOOST (global direct multi-horizon model)
# ============================================================

from numpy.lib.stride_tricks import sliding_window_view

import xgboost as xgb


# ------------------------------------------------------------
# XGBOOST SETTINGS
# ------------------------------------------------------------

XGB_PARAMS = {
    "n_estimators":          3000,
    "learning_rate":         0.05,
    "max_depth":             4,
    "min_child_weight":      1,
    "subsample":             0.8,
    "colsample_bytree":      0.8,
    "reg_lambda":            1.0,
    "objective":             "reg:squarederror",
    "tree_method":           "hist",
    "eval_metric":           "rmse",
    "early_stopping_rounds": 100,
    "random_state":          SEED,
    "n_jobs":                -1
}


LAG_COLS = [
    f"{variable}_lag{lag}"
    for variable in VARIABLES
    for lag in range(WINDOW)
]

XGB_FEATURES = (
    LAG_COLS
    + ["horizon", "target_month_sin", "target_month_cos"]
    + STATIC_COLS
)


# ------------------------------------------------------------
# Direct multi-horizon samples
# ------------------------------------------------------------

def make_direct_samples(frame, window, horizon):
    """
    One row per grid, origin month and horizon h:
        features = lags 0..window-1 before the origin
        target   = GHI at origin + h
    """

    tables = []

    for grid_number, group in frame.groupby("grid_number"):

        n = len(group)

        dates = group["date"].to_numpy()

        ghi = group[GHI_COL].to_numpy()

        # Row j covers months j ... j+window-1; reversed so column 0 = lag 0
        lags = np.hstack([
            sliding_window_view(group[variable].to_numpy(), window)[:, ::-1]
            for variable in VARIABLES
        ])

        origin_index = np.arange(window - 1, n)

        for h in range(1, horizon + 1):

            valid = origin_index + h < n

            origins = origin_index[valid]

            table = pd.DataFrame(lags[valid], columns=LAG_COLS)

            table.insert(0, "grid_number", grid_number)

            table["origin_date"] = dates[origins]

            table["target_date"] = dates[origins + h]

            table["horizon"] = h

            for column in STATIC_COLS:
                table[column] = group[column].iloc[0]

            table["target"] = ghi[origins + h]

            tables.append(table)

    samples = pd.concat(tables, ignore_index=True)

    target_month = samples["target_date"].dt.month

    samples["target_month_sin"] = np.sin(2 * np.pi * target_month / 12)

    samples["target_month_cos"] = np.cos(2 * np.pi * target_month / 12)

    return samples


samples = make_direct_samples(data, WINDOW, HORIZON)

print("\n" + "=" * 65)
print("XGBOOST: ROLLING VALIDATION")
print("=" * 65)

print(f"Samples built: {len(samples)}   features: {len(XGB_FEATURES)}")


# ------------------------------------------------------------
# Rolling validation
# ------------------------------------------------------------

xgb_predictions = []

for rnd in ROUNDS.itertuples(index=False):

    early_stop_start = rnd.train_end_date - pd.DateOffset(months=11)

    fit_rows = samples["target_date"] < early_stop_start

    es_rows = (
        (samples["target_date"] >= early_stop_start)
        & (samples["target_date"] <= rnd.train_end_date)
    )

    val_rows = samples["origin_date"] == rnd.train_end_date


    model = xgb.XGBRegressor(**XGB_PARAMS)

    model.fit(
        samples.loc[fit_rows, XGB_FEATURES],
        samples.loc[fit_rows, "target"],
        eval_set=[(samples.loc[es_rows, XGB_FEATURES], samples.loc[es_rows, "target"])],
        verbose=False
    )

    predictions = samples.loc[val_rows, ["grid_number", "target_date"]].copy()

    predictions["round"] = rnd.round

    predictions["predicted"] = model.predict(samples.loc[val_rows, XGB_FEATURES])

    xgb_predictions.append(predictions)

    model.save_model(MODEL_DIR / f"xgboost_round{rnd.round}.json")

    print(
        f"Round {rnd.round} (validate {rnd.validation_year}): "
        f"fit {fit_rows.sum()}  early-stop {es_rows.sum()}  "
        f"best iteration {model.best_iteration}  early-stop RMSE {model.best_score:.5f}"
    )


# ------------------------------------------------------------
# Feature importance of the last round
# ------------------------------------------------------------

importance = (
    pd.Series(
        model.get_booster().get_score(importance_type="gain"),
        name="gain"
    )
    .reindex(XGB_FEATURES)
    .fillna(0)
    .sort_values(ascending=False)
)

importance.to_csv(
    MODEL_DIR / "xgboost_feature_importance_last_round.csv",
    index_label="feature"
)

print("\nTop 10 features, last round (gain):")

print(importance.head(10).round(4).to_string())

xgb_results = save_rolling_results(
    pd.concat(xgb_predictions, ignore_index=True),
    "xgboost",
    "XGBoost"
)


XGBOOST: ROLLING VALIDATION
Samples built: 221526   features: 42
Round 1 (validate 2015): fit 100998  early-stop 13392  best iteration 1092  early-stop RMSE 0.42644
Round 2 (validate 2016): fit 114390  early-stop 13392  best iteration 1496  early-stop RMSE 0.50433
Round 3 (validate 2017): fit 127782  early-stop 13392  best iteration 932  early-stop RMSE 0.46413
Round 4 (validate 2018): fit 141174  early-stop 13392  best iteration 140  early-stop RMSE 0.37058
Round 5 (validate 2019): fit 154566  early-stop 13392  best iteration 1508  early-stop RMSE 0.42554
Round 6 (validate 2020): fit 167958  early-stop 13392  best iteration 1688  early-stop RMSE 0.42847
Round 7 (validate 2021): fit 181350  early-stop 13392  best iteration 415  early-stop RMSE 0.43454
Round 8 (validate 2022): fit 194742  early-stop 13392  best iteration 395  early-stop RMSE 0.39190

Top 10 features, last round (gain):
target_month_cos       441.2774
target_month_sin       269.1719
latitude               201.4814
longi

## 4. Reference forecasts and model comparison

Two reference forecasts use the same rounds:
- **Climatology:** the mean GHI of each calendar month over the round's training years. This is the standard benchmark for year-ahead forecasts.
- **Seasonal naive:** the GHI of the same month in the previous year.

**Skill score** = `1 − RMSE_model / RMSE_climatology`. A value above 0 means the model beats climatology.

Combined outputs:
- `rolling_metrics_by_grid_all_models.csv`: RMSE, MAE and nRMSE for every model, round and grid point
- `rolling_metrics_by_round_all_models.csv`: RMSE, MAE and nRMSE for every model and round
- `rolling_metrics_summary.csv`: each model over all rounds

In [5]:
# ============================================================
# 4. REFERENCE FORECASTS AND MODEL COMPARISON
# ============================================================

# ------------------------------------------------------------
# Climatology and seasonal naive
# ------------------------------------------------------------

climatology_predictions = []

naive_predictions = []

for rnd in ROUNDS.itertuples(index=False):

    train = data[data["date"] <= rnd.train_end_date]

    targets = data.loc[
        (data["date"] >= rnd.validation_start) & (data["date"] <= rnd.validation_end),
        ["grid_number", "date", "month"]
    ].rename(columns={"date": "target_date"})

    monthly_mean = (
        train.groupby(["grid_number", "month"], as_index=False)[GHI_COL]
        .mean()
        .rename(columns={GHI_COL: "predicted"})
    )

    previous_year = (
        train.loc[train["date"].dt.year == rnd.validation_year - 1, ["grid_number", "month", GHI_COL]]
        .rename(columns={GHI_COL: "predicted"})
    )

    climatology_predictions.append(
        targets.merge(monthly_mean, on=["grid_number", "month"]).assign(round=rnd.round)
    )

    naive_predictions.append(
        targets.merge(previous_year, on=["grid_number", "month"]).assign(round=rnd.round)
    )


save_rolling_results(
    pd.concat(climatology_predictions, ignore_index=True),
    "climatology",
    "Climatology"
)

save_rolling_results(
    pd.concat(naive_predictions, ignore_index=True),
    "seasonal_naive",
    "Seasonal naive"
)


# ------------------------------------------------------------
# Combine every model that has been run
# ------------------------------------------------------------

MODELS = {
    "climatology":    "Climatology",
    "seasonal_naive": "Seasonal naive",
    "sarimax":        "SARIMAX",
    "lstm":           "LSTM",
    "xgboost":        "XGBoost"
}

all_predictions, all_by_grid, all_by_round = [], [], []

for name, label in MODELS.items():

    path = PREDICTION_DIR / f"{name}_rolling_predictions.csv"

    if not path.exists():
        print(f"Skipping {label}: run its section first")
        continue

    all_predictions.append(pd.read_csv(path, parse_dates=["target_date"]))

    all_by_grid.append(pd.read_csv(METRICS_DIR / f"{name}_rolling_metrics_by_grid.csv"))

    all_by_round.append(pd.read_csv(METRICS_DIR / f"{name}_rolling_metrics_by_round.csv"))


all_predictions = pd.concat(all_predictions, ignore_index=True)

all_by_grid = pd.concat(all_by_grid, ignore_index=True)

all_by_round = pd.concat(all_by_round, ignore_index=True)

all_by_grid.to_csv(METRICS_DIR / "rolling_metrics_by_grid_all_models.csv", index=False)

all_by_round.to_csv(METRICS_DIR / "rolling_metrics_by_round_all_models.csv", index=False)


# ------------------------------------------------------------
# Summary over all rounds
# ------------------------------------------------------------

model_order = [label for label in MODELS.values() if label in set(all_predictions["model"])]

summary = (
    summarise_errors(all_predictions, ["model"])
    .set_index("model")
    .reindex(model_order)
)

round_rmse = all_by_round.pivot(index="validation_year", columns="model", values="RMSE")[model_order]

summary["round_RMSE_mean"] = round_rmse.mean()

summary["round_RMSE_std"] = round_rmse.std()

summary["skill_vs_climatology"] = 1 - summary["RMSE"] / summary.loc["Climatology", "RMSE"]

summary["rounds_beating_climatology"] = (
    round_rmse.lt(round_rmse["Climatology"], axis=0).sum()
)

grid_rmse = all_by_grid.pivot_table(
    index=["round", "grid_number"], columns="model", values="RMSE"
)[model_order]

summary["grid_rounds_beating_climatology_pct"] = (
    grid_rmse.lt(grid_rmse["Climatology"], axis=0).mean() * 100
)

summary.to_csv(METRICS_DIR / "rolling_metrics_summary.csv")


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\n" + "=" * 65)
print(
    f"ROLLING VALIDATION SUMMARY ({FIRST_VALIDATION_YEAR}-{LAST_VALIDATION_YEAR}, "
    f"{len(ROUNDS)} rounds, year-ahead)"
)
print("=" * 65)

print("\nRMSE by validation year (kWh/m²/day):")

print(round_rmse.round(4).to_string())

print("\nAll rounds:")

print(
    summary[[
        "RMSE", "MAE", "nRMSE_pct", "round_RMSE_std",
        "skill_vs_climatology", "rounds_beating_climatology",
        "grid_rounds_beating_climatology_pct"
    ]]
    .round(4)
    .to_string()
)

print(f"\nSaved: {METRICS_DIR / 'rolling_metrics_by_grid_all_models.csv'}")
print(f"Saved: {METRICS_DIR / 'rolling_metrics_by_round_all_models.csv'}")
print(f"Saved: {METRICS_DIR / 'rolling_metrics_summary.csv'}")


Climatology: rolling validation by round (all grids pooled)
 round  validation_year   RMSE    MAE  nRMSE_pct
     1             2015 0.5158 0.4391     9.6413
     2             2016 0.5092 0.3928     9.6203
     3             2017 0.3293 0.2654     6.5971
     4             2018 0.4199 0.3217     8.1363
     5             2019 0.5084 0.4306     9.5049
     6             2020 0.4523 0.3676     8.5973
     7             2021 0.3785 0.3017     7.5123
     8             2022 0.4301 0.3402     8.6016
All rounds: RMSE 0.4474   MAE 0.3574   nRMSE 8.64%
Saved: outputs\standalone_models\rolling_predictions\climatology_rolling_predictions.csv
Saved: outputs\standalone_models\rolling_metrics\climatology_rolling_metrics_by_grid.csv
Saved: outputs\standalone_models\rolling_metrics\climatology_rolling_metrics_by_round.csv

Seasonal naive: rolling validation by round (all grids pooled)
 round  validation_year   RMSE    MAE  nRMSE_pct
     1             2015 0.5395 0.4323    10.0841
     2           

## References

The code was written for this study using the official APIs of the libraries below. It was not adapted from third-party code. Exact library versions are saved in `outputs/standalone_models/environment.txt`.

**Software libraries**
- Seabold, S., & Perktold, J. (2010). Statsmodels: Econometric and statistical modeling with Python. *Proceedings of the 9th Python in Science Conference*, 92–96. https://doi.org/10.25080/Majora-92bf1922-011. SARIMAX: https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.sarimax.SARIMAX.html
- Paszke, A., Gross, S., Massa, F., Lerer, A., Bradbury, J., Chanan, G., … Chintala, S. (2019). PyTorch: An imperative style, high-performance deep learning library. *Advances in Neural Information Processing Systems 32*, 8024–8035. `torch.nn.LSTM`: https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html
- Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785–794. https://doi.org/10.1145/2939672.2939785. Python API: https://xgboost.readthedocs.io/en/stable/python/python_api.html
- Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, É. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. `MinMaxScaler`
- Harris, C. R., Millman, K. J., van der Walt, S. J., et al. (2020). Array programming with NumPy. *Nature, 585*, 357–362. https://doi.org/10.1038/s41586-020-2649-2
- McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a
- Joblib Development Team. (2024). *Joblib: Running Python functions as pipeline jobs*. https://joblib.readthedocs.io

**Methods**
- Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: An analysis and review. *International Journal of Forecasting, 16*(4), 437–450. https://doi.org/10.1016/S0169-2070(00)00065-0 (rolling-origin evaluation)
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and practice* (3rd ed.). OTexts. https://otexts.com/fpp3/ (time-series cross-validation, climatology/seasonal naive benchmarks, skill scores)
- Ben Taieb, S., Bontempi, G., Atiya, A. F., & Sorjamaa, A. (2012). A review and comparison of strategies for multi-step ahead time series forecasting based on the NN5 forecasting competition. *Expert Systems with Applications, 39*(8), 7067–7083. https://doi.org/10.1016/j.eswa.2012.01.039 (direct and multi-output strategies)
- Box, G. E. P., Jenkins, G. M., Reinsel, G. C., & Ljung, G. M. (2015). *Time series analysis: Forecasting and control* (5th ed.). Wiley. (SARIMA modelling)
- Dickey, D. A., & Fuller, W. A. (1979). Distribution of the estimators for autoregressive time series with a unit root. *Journal of the American Statistical Association, 74*(366a), 427–431. https://doi.org/10.1080/01621459.1979.10482531 (ADF test)
- Akaike, H. (1974). A new look at the statistical model identification. *IEEE Transactions on Automatic Control, 19*(6), 716–723. https://doi.org/10.1109/TAC.1974.1100705 (AIC)
- Ljung, G. M., & Box, G. E. P. (1978). On a measure of lack of fit in time series models. *Biometrika, 65*(2), 297–303. https://doi.org/10.1093/biomet/65.2.297 (Ljung-Box test)
- Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation, 9*(8), 1735–1780. https://doi.org/10.1162/neco.1997.9.8.1735
- Kingma, D. P., & Ba, J. (2015). Adam: A method for stochastic optimization. *3rd International Conference on Learning Representations (ICLR)*. https://arxiv.org/abs/1412.6980